# Data Science for Business - Logit on Micro Mortgages

*Session 4 · ISLP 4.3 · 4.4.2 (Tables 4.4–4.7, Fig. 4.8 ROC) · 5.1.3, 5.1.5 · Labs 4.7.2, 5.3.2*

## Initialize notebook
Load required packages. Set up workspace, e.g., set theme for plotting and initialize the random number generator.

In [ ]:
import numpy as np
import pandas as pd

from matplotlib import pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, RocCurveDisplay, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import statsmodels.formula.api as smf

In [ ]:
np.random.seed(42)
plt.style.use('fivethirtyeight')

## Problem description

In India, there are about 20 million home loan (mortgage) aspirants
working in the informal sector:

- Monthly income between INR 20,000-25,000 (\$ 325-400)
- Typically no formal accounts and documents (e.g., tax returns, income proofs, bank statements)
- Often use services of money lenders with interest rates between 30 and 60% per annum

Providing mortgages to this group of customers requires to quickly and
efficiently assess their creditworthiness. Due to a lack of formal
documents and objective data, most financial institutions perform
interview-based processes to decide about these loan requests:

Strength of the current process:

-   Interview-based field assessment

-   Relaxation of document requirements

Weaknesses of the current process:

-   Costly (total transaction costs as high as 30% of loan volume)

-   Subjective judgments; depends on individual skills and motivations

-   Low reliability across branches and credit officers

-   Risk of corruption and fraud

Our goal: predict the credit officers' decision (`Decision`: 1 = approved, 0 = rejected) from the application data. See `variables.pdf` for a description of the variables.

## Load data

Load data from CSV file.

In [ ]:
data = pd.read_csv('https://raw.githubusercontent.com/olivermueller/ds4b-2026/main/Session_04/micromortgage.csv')

In [ ]:
data.head()

## Prepare data

We do some minimal data preparation, that is, drop the `ID` column and turn the `Tier` variable into a string, so that it is treated as categorical.

In [ ]:
data = data.drop(['ID'], axis=1)
data["Tier"] = "T" + data["Tier"].astype(str)

Next, we perform a standard 80:20 random train/test split (validation set approach, ISLP 5.1.1). `stratify` keeps the share of approvals equal in both sets.

In [ ]:
train, test = train_test_split(data, test_size=0.2, random_state=42, stratify=data["Decision"])

## Exploratory data analysis

### Descriptive summary statistics

Calculate the base rate of mortgage approvals.

In [ ]:
train["Decision"].mean()

About 80% of applications are approved. That's the benchmark a useful model must beat: always predicting "approve" already gives 80% accuracy.

Note that here class 1 is *approval*, whereas the positive class in ISLP Ch. 4 is *default*. Keep this in mind when we discuss precision and recall below.

### Relationships between response and predictors

In [ ]:
sns.boxplot(data=train, x="Decision", y="Age")
plt.show()

**Your task:** Plot `TotInc` (total income) against `Decision`.

In [ ]:
# YOUR CODE HERE

In [ ]:
sns.barplot(data=train, x="Gender", y="Decision")
plt.show()

**Your task:** Compare approval rates by `Tier` or `Employment_Type`.

In [ ]:
# YOUR CODE HERE

## Fit model

Ok, now we are ready to fit our first logistic regression model (ISLP 4.3). We use `Age`, `Gender`, `Dwnpay` (down payment), `LoanReq` (requested loan amount) and `Term` as predictors, plus the interaction between `LoanReq` and `Term`. `Gender` is coded with `Female` as the baseline (alphabetically first).

In [ ]:
model_logit = smf.logit(formula='Decision ~ Age + Gender + Dwnpay + LoanReq * Term', data=train)
model_logit = model_logit.fit()

In [ ]:
print(model_logit.summary())

The coefficients are estimated by maximum likelihood (ISLP 4.3.2) and change the *log-odds* of approval (ISLP eq. 4.4). $e^{\beta}$ is the odds ratio: the factor by which the odds of approval change per unit increase of the predictor. With the interaction, the effect of `LoanReq` depends on `Term`. (The ISLP lab 4.7.2 uses `sm.GLM(..., family=sm.families.Binomial())`; it gives the same estimates.)

In [ ]:
np.exp(model_logit.params)

## Make predictions

Let's compute predictions, both in the form of probabilities and binary decisions. We classify an application as approved if the predicted probability is at least 0.5. A bank would pick the threshold from the cost of a wrong approval vs. a wrong rejection (ISLP 4.4.2).

In [ ]:
threshold = 0.5
pred_proba = model_logit.predict(test)
pred_label = (pred_proba >= threshold).astype(int)

In [ ]:
pred_proba

In [ ]:
pred_label

## Evaluate accuracy

We begin by calculating some standard metrics like precision, recall, F1-score, and accuracy. Remember that the row for class 1 refers to *approvals*.

`zero_division=0` reports a precision of 0 instead of a warning if the model never predicts a class.

In [ ]:
print(classification_report(test["Decision"], pred_label, zero_division=0))

With a threshold of 0.5, our model approves (almost) every application: its accuracy is no better than the base rate, and the recall for rejections (class 0) is close to zero. The predicted probabilities do carry some information, though; the ROC curve below evaluates them over all possible thresholds.

The confusion matrix helps us to diagnose what kind of mistakes the model makes. Careful: scikit-learn puts the **true** classes in the rows and the predicted classes in the columns, i.e., it is transposed compared to ISLP Tables 4.4 and 4.6.

In [ ]:
ConfusionMatrixDisplay.from_predictions(test["Decision"], pred_label)
plt.grid(False)
plt.show()

Finally, we plot the ROC curve and compute the AUC, which characterizes the performance of a classifier over all thresholds in one single number.

In [ ]:
RocCurveDisplay.from_predictions(test["Decision"], pred_proba, name="Logistic regression")
plt.show()

print("Test AUC:", roc_auc_score(test["Decision"], pred_proba))

The ROC curve traces the true positive rate (TPR, sensitivity) against the false positive rate (FPR, 1 − specificity) over all thresholds; AUC 0.5 = guessing, 1 = perfect (ISLP Fig. 4.8, Table 4.7).

## Cross-validation

Instead of a single train/test split, we can use $k$-fold cross-validation (ISLP 5.1.3, 5.1.5) to evaluate model performance. This gives a more robust estimate of how well a model generalizes and also shows the variation across folds. We run it on the training data only, so the test set stays untouched.

Note that this is a different model than above: here we use a set of numerical loan characteristics as predictors, and scikit-learn instead of statsmodels:
- `LogisticRegression` in scikit-learn is L2-penalised by default (`C=1`, the inverse of the penalty λ). A huge `C` (here `C=1e10`) means practically no penalty, i.e., the plain logistic regression of ISLP Ch. 4. ISLP Lab 4.7.6 uses the same trick.
- The predictors are on very different scales (rupees, months, percentages), so we standardize them. Putting `StandardScaler` into a pipeline ensures it is fitted on the training folds only, within each CV split.
- `StratifiedKFold` keeps the approval rate equal across folds; `shuffle=True` with `random_state=42` makes the folds random but reproducible.
- `scoring='roc_auc'` computes the AUC on each held-out fold.

In [ ]:
cv_data = train[['LoanReq', 'Term', 'Dwnpay', 'BankSave', 'CalcEmi', 'IIR', 'IAR', 'FOIR', 'LTV', 'LVR', 'Decision']]
X_cv = cv_data.drop(columns=['Decision'])
y_cv = cv_data['Decision']

# C = 1/λ; a huge C means practically no penalty (as in ISLP Lab 4.7.6)
model_cv = make_pipeline(StandardScaler(), LogisticRegression(C=1e10, max_iter=1000))
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(model_cv, X_cv, y_cv, cv=folds, scoring='roc_auc')

print(f'Cross-validated AUC scores: {cv_scores}')
print(f'Mean AUC score: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

Compare the mean cross-validated AUC with the test AUC of our first model: the loan characteristics (e.g., the ratios `FOIR` and `LTV`) seem to carry much more information about the decision than age and gender.

## Your turn!

**Your task:** Improve the logistic regression model from *Fit model* by including more predictors (e.g., `Tier`, `Employment_Type`, `TotInc`, `LTV`). Did the extra predictors improve the test AUC, or did you overfit?

In [ ]:
# YOUR CODE HERE